# CSE 151B Improvements Notebook

Builds on the initial notebook: sampling sweep, self-consistency, GRPO LoRA. Final cell runs the pipeline on `data/private.jsonl`.


## 1. Environment Setup

Comment out after first install. Restart kernel.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

!~/.local/bin/uv venv .venv --seed --python 3.11

!~/.local/bin/uv pip install --python .venv/bin/python \
    sympy numpy 'transformers>=4.51,<5' 'vllm>=0.8.5' 'torch>=2.5' \
    tqdm bitsandbytes antlr4-python3-runtime==4.11.1 \
    ipykernel jupyter trl peft accelerate datasets

!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel.")

In [ ]:
!source ./.venv/bin/activate

## 2. Imports & Configuration

Both A40s by default (TP=2 vLLM, accelerate for GRPO). Override with `GPU_IDS` / `TENSOR_PARALLEL`.

In [ ]:
import json, os, re, sys, time, statistics, gc
from pathlib import Path
from collections import Counter

MODEL_ID         = "Qwen/Qwen3-4B-Thinking-2507"
GPU_IDS          = os.environ.get("GPU_IDS", "0,1")
TENSOR_PARALLEL  = int(os.environ.get("TENSOR_PARALLEL", "2"))
DATA_PATH        = "../data/public.jsonl"
PRIVATE_PATH     = "../data/private.jsonl"
FILTERED_PATH    = "../data/public_filtered.jsonl"
RESULTS_DIR      = Path("../results")
LOG_DIR          = RESULTS_DIR / "logs"
MODELS_DIR       = Path("../models")
MAX_TOKENS       = 20000
MAX_MODEL_LEN    = 24576

EVAL_HEAD_N      = 50
EVAL_SKIP_IDS    = {0, 1, 8, 12, 13, 14}

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS

for d in (RESULTS_DIR, LOG_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

from transformers import AutoTokenizer
from tqdm import tqdm
from vllm import LLM, SamplingParams

## 3. Load Dataset + Eval Slice

First 50 items minus the flagged IDs. Same slice across every step.

In [ ]:
data = [json.loads(l) for l in open(DATA_PATH)]
print(f"Loaded {len(data)} questions")

eval_items = [d for d in data[:EVAL_HEAD_N] if d.get("id") not in EVAL_SKIP_IDS]
n_mcq_eval = sum(bool(d.get("options")) for d in eval_items)
print(f"Eval slice: {len(eval_items)} items ({n_mcq_eval} MCQ, {len(eval_items)-n_mcq_eval} free-form)")
print(f"  skipped IDs: {sorted(EVAL_SKIP_IDS)}")

## 4. Prompt Construction

Same prompts as the initial notebook.


In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician solving competition problems across algebra, "
    "calculus, statistics, probability, linear algebra, geometry, number theory, "
    "and discrete math. Reason carefully, then emit a final answer the automated "
    "grader can parse.\n"
    "\n"
    "# Method\n"
    "- Identify the problem type and the most direct solution method.\n"
    "- State the key formula or theorem before computing.\n"
    "- Keep every quantity in EXACT symbolic form throughout. Do not round, "
    "truncate, or convert to decimal at any intermediate step.\n"
    "- Verify by substitution, a limit/edge case, or solving a simpler instance "
    "before committing. If a check fails, restart from the broken step.\n"
    "\n"
    "# Final-answer format (STRICT — grader uses regex + sympy with 1e-8 relative tolerance)\n"
    "- Place the final answer inside \\boxed{...} at the very end of your response.\n"
    "- ALWAYS prefer exact symbolic forms over decimals.\n"
    "    Use \\boxed{\\frac{1}{3}}    NOT \\boxed{0.333}\n"
    "    Use \\boxed{\\sqrt{2}}      NOT \\boxed{1.414}\n"
    "    Use \\boxed{\\frac{\\pi}{4}} NOT \\boxed{0.7854}\n"
    "    Use \\boxed{\\ln 2}         NOT \\boxed{0.6931}\n"
    "    Use \\boxed{e^{2}}          NOT \\boxed{7.389}\n"
    "- Decimal only if the exact value is itself a finite decimal or the problem asks for one. "
    "Never write \"\\approx\" inside the box.\n"
    "- Inside \\boxed{}: bare value(s) only. No units, no \"x =\", no \\text{...}.\n"
    "\n"
    "# Multiple sub-answers\n"
    "Put ALL sub-answers in a SINGLE \\boxed{} separated by commas, in order:\n"
    "    \\boxed{580, 660, 80}\n"
    "    \\boxed{\\frac{1}{2}, \\sqrt{3}, \\pi}\n"
    "Do not split sub-answers across multiple boxes."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician answering a multiple-choice competition "
    "problem. Solve rigorously, then output ONE letter.\n"
    "\n"
    "# Method\n"
    "- Compute the answer in EXACT symbolic form first.\n"
    "- Convert to decimal only at the very end to match against the listed options.\n"
    "- If your value matches no option, RECOMPUTE — don't pick the visually closest blindly.\n"
    "- Sanity check (sign, magnitude) before committing.\n"
    "\n"
    "# Final-answer format (STRICT)\n"
    "- Output exactly one \\boxed{X} where X is a single capital letter (A, B, C, ...).\n"
    "- Examples: \\boxed{C}, \\boxed{E}.\n"
    "- No \"Option C\", no extra characters — just the letter."
)

USER_PROMPT_SUFFIX = "\n\nPlease reason step by step, and put your final answer within \\boxed{}."


def build_prompt(question, options):
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user = f"{question}\n\nOptions:\n{opts_text}{USER_PROMPT_SUFFIX}"
        sys_p = SYSTEM_PROMPT_MCQ
    else:
        user = f"{question}{USER_PROMPT_SUFFIX}"
        sys_p = SYSTEM_PROMPT_MATH
    return sys_p, user


## 5. Scoring + Summary

MCQ: letter match. Free-form: `Judger.auto_judge`.


In [ ]:
def extract_letter(text):
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_one(item, response):
    is_mcq = bool(item.get("options"))
    gold = item["answer"]
    if is_mcq:
        return extract_letter(response) == str(gold).strip().upper()
    gold_list = gold if isinstance(gold, list) else [gold]
    try:
        return bool(judger.auto_judge(pred=response, gold=gold_list,
                                      options=[[]] * len(gold_list)))
    except Exception:
        return False

def print_summary(label, results):
    mcq = [r for r in results if r.get("is_mcq")]
    free = [r for r in results if not r.get("is_mcq")]
    def acc(rs): return sum(r["correct"] for r in rs) / len(rs) * 100 if rs else 0.0
    print("=" * 60)
    print(f"  {label}  ({len(results)} items)")
    print("=" * 60)
    print(f"  MCQ        : {sum(r['correct'] for r in mcq):4d} / {len(mcq):4d}  ({acc(mcq):.2f}%)")
    print(f"  Free-form  : {sum(r['correct'] for r in free):4d} / {len(free):4d}  ({acc(free):.2f}%)")
    print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
    print("=" * 60)

def write_jsonl(path, rows):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r, default=str) + "\n")

## 6. Load vLLM (BF16, TP=2)

Reused across steps 1-3 and the final run. `enable_lora=True` if an adapter is on disk.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Detect a saved GRPO adapter so we can enable LoRA at load time.
GRPO_ADAPTER_DIR = MODELS_DIR / "grpo_lora"
HAS_GRPO_ADAPTER = GRPO_ADAPTER_DIR.exists() and any(GRPO_ADAPTER_DIR.iterdir())

llm_kwargs = dict(
    model=MODEL_ID,
    dtype="bfloat16",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.75,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=128,
    max_num_batched_tokens=16384,
    tensor_parallel_size=TENSOR_PARALLEL,
)
if HAS_GRPO_ADAPTER:
    llm_kwargs.update(enable_lora=True, max_lora_rank=32, max_loras=1)

llm = LLM(**llm_kwargs)

if HAS_GRPO_ADAPTER:
    from vllm.lora.request import LoRARequest
    GRPO_LORA = LoRARequest("grpo", 1, str(GRPO_ADAPTER_DIR))
    print(f"Loaded with GRPO adapter from {GRPO_ADAPTER_DIR}")
else:
    GRPO_LORA = None
    print("No GRPO adapter found — running base model.")

## 7. STEP 1 - Sampling parameter sweep

24-config grid, 3 samples per question. Validates Qwen-recommended T=0.6/p=0.95/k=20/m=0.


In [ ]:
SWEEP_GRID = {
    "temperature": [0.3, 0.6, 0.9],
    "top_p":       [0.8, 0.95],
    "top_k":       [0, 20],
    "min_p":       [0.0, 0.05],
}
SAMPLES_PER_QUESTION = 3
SEED_BASE = 1000

def iter_configs():
    import itertools
    keys = list(SWEEP_GRID.keys())
    for vals in itertools.product(*[SWEEP_GRID[k] for k in keys]):
        yield dict(zip(keys, vals))

def cfg_key(c):
    return f"T{c['temperature']}_p{c['top_p']}_k{c['top_k']}_mp{c['min_p']}"

# Build prompts once.
sweep_prompts = []
for it in eval_items:
    sys_p, usr_p = build_prompt(it["question"], it.get("options"))
    sweep_prompts.append(tokenizer.apply_chat_template(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": usr_p}],
        tokenize=False, add_generation_prompt=True,
    ))

# Expand to (n_items × samples) prompts; vary seed per sample.
expanded_prompts, expanded_meta = [], []
for i, p in enumerate(sweep_prompts):
    for s in range(SAMPLES_PER_QUESTION):
        expanded_prompts.append(p)
        expanded_meta.append((i, s))

print(f"Sweep: {len(list(iter_configs()))} configs × {len(expanded_prompts)} prompts each")

In [ ]:
sweep_rows = []
sweep_path = RESULTS_DIR / "param_sweep.jsonl"

for cfg in iter_configs():
    key = cfg_key(cfg)
    t0 = time.time()
    sps = [SamplingParams(
            temperature=cfg["temperature"],
            top_p=cfg["top_p"],
            top_k=cfg["top_k"] if cfg["top_k"] > 0 else -1,
            min_p=cfg["min_p"],
            max_tokens=MAX_TOKENS,
            repetition_penalty=1.0,
            seed=SEED_BASE + s)
        for (_, s) in expanded_meta]

    outputs = llm.generate(expanded_prompts, sampling_params=sps,
                           lora_request=GRPO_LORA)

    per_item_correct = [[] for _ in eval_items]
    per_item_lens    = [[] for _ in eval_items]
    for (i, _), out in zip(expanded_meta, outputs):
        text = out.outputs[0].text
        per_item_correct[i].append(score_one(eval_items[i], text))
        per_item_lens[i].append(len(text))

    item_accs = [statistics.mean(map(float, cs)) for cs in per_item_correct]
    mcq_idx  = [j for j, it in enumerate(eval_items) if it.get("options")]
    free_idx = [j for j, it in enumerate(eval_items) if not it.get("options")]
    mcq_acc  = statistics.mean([item_accs[j] for j in mcq_idx]) * 100 if mcq_idx else 0.0
    free_acc = statistics.mean([item_accs[j] for j in free_idx]) * 100 if free_idx else 0.0
    overall  = statistics.mean(item_accs) * 100

    sweep_rows.append(dict(
        config_key=key, config=cfg,
        mcq_acc=mcq_acc, free_acc=free_acc, overall_acc=overall,
        mean_response_len=statistics.mean([statistics.mean(ls) for ls in per_item_lens]),
        wall_seconds=round(time.time() - t0, 2),
    ))
    print(f"[sweep] {key:34s}  mcq={mcq_acc:5.2f}  free={free_acc:5.2f}  overall={overall:5.2f}")
    write_jsonl(sweep_path, sweep_rows)

sweep_rows.sort(key=lambda r: -r["overall_acc"])
winning_cfg = sweep_rows[0]["config"]
print(f"\nWINNER: {sweep_rows[0]['config_key']}  overall={sweep_rows[0]['overall_acc']:.2f}%")

## 8. STEP 2 - Self-consistency (k=3)

Run k=3 samples per item, vote on the modal letter (MCQ) or modal judger-normalized form (free-form). Tie-break by first occurrence.


In [ ]:
SC_K = 3

def vote_key_free(text):
    extracted = judger.extract_ans(text) or ""
    parts = [judger.norm_ans_str(p) for p in judger.split_by_comma(extracted)]
    if not parts:        return ""
    if len(parts) == 1:  return parts[0]
    return "(" + ", ".join(parts) + ")"

def vote_key_mcq(text):
    return extract_letter(text)

def modal(keys):
    counts = Counter(k for k in keys if k)
    if not counts: return ""
    top = max(counts.values())
    winners = [k for k, c in counts.items() if c == top]
    for k in keys:
        if k in winners:
            return k
    return winners[0]


def generate(item, lora_request=None):
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    messages = [{"role": "system", "content": sys_p},
                {"role": "user", "content": usr_p}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    n_prompt = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
    max_new = MAX_MODEL_LEN - n_prompt - 64
    if max_new < 256:
        return ""
    sp = SamplingParams(
        temperature=SAMPLING_CFG.get("temperature", 0.6),
        top_p=SAMPLING_CFG.get("top_p", 0.95),
        top_k=SAMPLING_CFG.get("top_k", 20) if SAMPLING_CFG.get("top_k", 20) > 0 else -1,
        min_p=SAMPLING_CFG.get("min_p", 0.0),
        max_tokens=max_new,
        repetition_penalty=1.0,
    )
    out = llm.generate([prompt], sampling_params=sp, lora_request=lora_request)
    return out[0].outputs[0].text


def run_sc_one(item, k=SC_K, lora_request=None):
    is_mcq = bool(item.get("options"))
    responses, keys = [], []
    for _ in range(k):
        try:
            text = generate(item, lora_request=lora_request)
        except Exception as e:
            text = f"[error: {e!r}]"
        responses.append(text)
        keys.append(vote_key_mcq(text) if is_mcq else vote_key_free(text))
    modal_key = modal(keys)
    chosen = next((r for r, kk in zip(responses, keys) if kk == modal_key),
                  responses[0] if responses else "")
    return chosen, modal_key, keys


In [ ]:
sc_rows = []
sc_path = RESULTS_DIR / "sc_k8_results.jsonl"

for idx, item in enumerate(tqdm(eval_items, desc=f"SC k={SC_K}")):
    chosen, modal_key, keys = run_sc_one(item, k=SC_K, lora_request=GRPO_LORA)
    sc_rows.append(dict(
        id=item.get("id"), is_mcq=bool(item.get("options")),
        gold=item["answer"], response=chosen,
        correct=score_one(item, chosen), k=SC_K,
        sample_keys=keys, modal_key=modal_key,
    ))
    if (idx + 1) % 3 == 0 or idx == len(eval_items) - 1:
        write_jsonl(sc_path, sc_rows)

write_jsonl(sc_path, sc_rows)
print_summary(f"SC k={SC_K}", sc_rows)

## 9. STEP 3 - GRPO LoRA fine-tune

Free vLLM, write `notebooks/grpo_train.py`, `accelerate launch --num_processes=2`, reload vLLM with `enable_lora=True`.


In [ ]:
# Step 4a — free the inference vLLM so the GRPO trainer + rollout vLLM can claim the GPU.
try:
    del llm
except NameError:
    pass
try:
    from vllm.distributed.parallel_state import destroy_distributed_environment, destroy_model_parallel
    destroy_model_parallel()
    destroy_distributed_environment()
except Exception as e:
    print(f"(vllm distributed cleanup skipped: {e})")
gc.collect()
import torch
torch.cuda.empty_cache()
print("vLLM freed.")

In [ ]:
# Step 4b — write the GRPO training script. The cell IS the file content
# (so it stays inside the notebook for submission). After this cell runs,
# notebooks/grpo_train.py exists on disk and accelerate can launch it.

# Note: triple-double-quoted raw string keeps backslashes literal so regex
# patterns like \boxed survive intact when written to disk.
GRPO_TRAIN_SOURCE = r"""
import json, os, re, sys
from pathlib import Path

import torch

# peft 0.19.1 unconditionally imports EmbeddingParallel from transformers,
# but transformers 4.57.6 doesn't export it. The import is in a TP-only code
# path that we never hit, but the import statement itself fails. Add a benign
# alias before peft is loaded so the import succeeds.
import transformers.integrations.tensor_parallel as _tp
if not hasattr(_tp, 'EmbeddingParallel'):
    _tp.EmbeddingParallel = _tp.ReplicateParallel

from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

REPO = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO / 'notebooks'))
from judger import Judger

MODEL_ID      = 'Qwen/Qwen3-4B-Thinking-2507'
FILTERED_PATH = REPO / 'data' / 'public_filtered.jsonl'
OUTPUT_DIR    = REPO / 'models' / 'grpo_lora'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRPO_MAX_PROMPT_TOKENS = 2048
SAMPLING_CFG = {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20, 'min_p': 0.0}

judger = Judger(strict_extract=False)

# Prompts MUST match the notebook (otherwise reward signal is biased).
SYSTEM_PROMPT_MATH = __SYSTEM_PROMPT_MATH__
SYSTEM_PROMPT_MCQ  = __SYSTEM_PROMPT_MCQ__
USER_PROMPT_SUFFIX = __USER_PROMPT_SUFFIX__


def build_prompt(question, options):
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = '\n'.join(f'{lbl}. {opt.strip()}' for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f'{question}\n\nOptions:\n{opts_text}{USER_PROMPT_SUFFIX}'
    return SYSTEM_PROMPT_MATH, f'{question}{USER_PROMPT_SUFFIX}'


def extract_letter(text):
    m = re.search(r'\\boxed\{([A-Za-z])\}', text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r'\b([A-Z])\b', text.upper())
    return matches[-1] if matches else ''


def score_one(item, response):
    if item.get('options'):
        return extract_letter(response) == str(item['answer']).strip().upper()
    gold = item['answer']
    gold_list = gold if isinstance(gold, list) else [gold]
    try:
        return bool(judger.auto_judge(pred=response, gold=gold_list,
                                      options=[[]] * len(gold_list)))
    except Exception:
        return False


def reward_fn(prompts, completions, **kwargs):
    items = [json.loads(j) for j in kwargs['item_json']]
    rewards = []
    for comp, item in zip(completions, items):
        text = comp if isinstance(comp, str) else (
            comp[0]['content'] if isinstance(comp, list) and comp else str(comp)
        )
        try:
            rewards.append(1.0 if score_one(item, text) else 0.0)
        except Exception:
            rewards.append(0.0)
    return rewards


def main():
    if not FILTERED_PATH.exists():
        raise FileNotFoundError(f'Add {FILTERED_PATH} before running GRPO.')

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    rows = [json.loads(l) for l in open(FILTERED_PATH)]
    print(f'Filtered set: {len(rows)} items')

    def to_record(item):
        sys_p, usr_p = build_prompt(item['question'], item.get('options'))
        prompt = tokenizer.apply_chat_template(
            [{'role': 'system', 'content': sys_p}, {'role': 'user', 'content': usr_p}],
            tokenize=False, add_generation_prompt=True,
        )
        ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
        if len(ids) > GRPO_MAX_PROMPT_TOKENS:
            prompt = tokenizer.decode(ids[-GRPO_MAX_PROMPT_TOKENS:], skip_special_tokens=False)
        return dict(prompt=prompt, item_json=json.dumps(item))

    dataset = Dataset.from_list([to_record(r) for r in rows])

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    # device_map pins the 4-bit model to this rank's GPU. accelerate sets
    # LOCAL_RANK; without it both ranks would land on GPU 0 and DDP errors.
    local_rank = int(os.environ.get('LOCAL_RANK', '0'))
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb,
        torch_dtype=torch.bfloat16, trust_remote_code=True,
        device_map={'': local_rank},
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    peft_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.0, bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    )

    grpo_cfg = GRPOConfig(
        output_dir=str(OUTPUT_DIR),
        learning_rate=1e-6,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        bf16=True,
        logging_steps=1,
        save_steps=25,
        max_grad_norm=1.0,
        beta=0.04,
        num_generations=4,
        # 2048 reduces rollout-time variance — long completions on one rank
        # were causing the other rank to wait forever in NCCL collectives.
        max_completion_length=2048,
        temperature=SAMPLING_CFG['temperature'],
        top_p=SAMPLING_CFG['top_p'],
        top_k=SAMPLING_CFG['top_k'],
        min_p=SAMPLING_CFG['min_p'],
        use_vllm=True,
        vllm_mode='colocate',
        vllm_gpu_memory_utilization=0.35,
        vllm_max_model_length=8192,
        report_to='none',
        ddp_find_unused_parameters=False,
    )

    trainer = GRPOTrainer(
        model=model, args=grpo_cfg,
        train_dataset=dataset, reward_funcs=[reward_fn],
        peft_config=peft_cfg, processing_class=tokenizer,
    )

    # resume_from_checkpoint=True picks up the latest checkpoint-N under OUTPUT_DIR
    # so retries continue from wherever the previous crash left off.
    resume = any(p.is_dir() and p.name.startswith('checkpoint-')
                 for p in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else False
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(str(OUTPUT_DIR))
    print(f'GRPO adapter saved to {OUTPUT_DIR}')


if __name__ == '__main__':
    main()
"""

# Substitute in the actual prompt strings so train-time prompts == eval-time prompts.
GRPO_TRAIN_SOURCE = (GRPO_TRAIN_SOURCE
    .replace("__SYSTEM_PROMPT_MATH__", repr(SYSTEM_PROMPT_MATH))
    .replace("__SYSTEM_PROMPT_MCQ__",  repr(SYSTEM_PROMPT_MCQ))
    .replace("__USER_PROMPT_SUFFIX__", repr(USER_PROMPT_SUFFIX))
)

Path("grpo_train.py").write_text(GRPO_TRAIN_SOURCE)
print(f"Wrote notebooks/grpo_train.py ({len(GRPO_TRAIN_SOURCE)} bytes)")

In [ ]:
# Step 4c — accelerate launch with retry loop AND a watchdog. The retry-only
# version of this cell deadlocked at step 55: one rank stuck inside vLLM
# rollout, the other waiting in NCCL allgather, no progress for 28 hours. The
# subprocess didn't exit so the retry never fired.
#
# Watchdog now monitors the GRPO log; if it doesn't grow for STUCK_LIMIT
# seconds we SIGKILL the whole process group so the retry loop can fire.

import os, signal, subprocess, threading, time
from pathlib import Path

env = {**os.environ,
       "CUDA_VISIBLE_DEVICES": GPU_IDS,
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
       "HF_HUB_DISABLE_PROGRESS_BARS": "1",
       "TORCH_NCCL_BLOCKING_WAIT": "1",
       "TORCH_NCCL_HEARTBEAT_TIMEOUT_SEC": "7200",
       "NCCL_TIMEOUT_MS": "7200000"}

MAX_GRPO_ATTEMPTS = 12
STUCK_LIMIT_SEC   = 900    # 15 min with no log growth ⇒ killed
WATCHDOG_POLL_SEC = 30

attempt_log = Path("../results/logs/grpo_attempt.log")
attempt_log.parent.mkdir(parents=True, exist_ok=True)

def watchdog(proc, log_path, max_stuck):
    last_size, last_change = 0, time.time()
    while proc.poll() is None:
        time.sleep(WATCHDOG_POLL_SEC)
        cur_size = log_path.stat().st_size if log_path.exists() else 0
        if cur_size > last_size:
            last_size, last_change = cur_size, time.time()
        elif time.time() - last_change > max_stuck:
            print(f"\n[watchdog] no log progress for {max_stuck//60} min — killing process group")
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
                time.sleep(10)
                if proc.poll() is None:
                    os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except ProcessLookupError:
                pass
            return

for attempt in range(1, MAX_GRPO_ATTEMPTS + 1):
    print(f"\n=== GRPO attempt {attempt}/{MAX_GRPO_ATTEMPTS} ===")
    # Truncate the attempt log so the watchdog only sees growth from THIS attempt.
    attempt_log.write_text("")

    with open(attempt_log, "wb") as logf:
        proc = subprocess.Popen(
            [".venv/bin/accelerate", "launch",
             "--multi_gpu", "--num_processes=2", "--num_machines=1",
             "--mixed_precision=bf16",
             "notebooks/grpo_train.py"],
            cwd="..", env=env,
            stdout=logf, stderr=subprocess.STDOUT,
            preexec_fn=os.setsid,    # new process group so we can killpg later
        )
        wd = threading.Thread(target=watchdog,
                              args=(proc, attempt_log, STUCK_LIMIT_SEC),
                              daemon=True)
        wd.start()
        rc = proc.wait()

    print(f"attempt {attempt} exit code: {rc}")
    # Surface the last ~25 lines of the attempt log so we have a breadcrumb.
    try:
        tail = attempt_log.read_text(errors="replace").splitlines()[-25:]
        for line in tail:
            print(line)
    except Exception:
        pass

    if rc == 0:
        print("GRPO training completed.")
        break
    time.sleep(60)
else:
    print(f"GRPO did not complete after {MAX_GRPO_ATTEMPTS} attempts. "
          "Will continue with whatever checkpoint exists.")

In [ ]:
# Step 4d — reload vLLM. If a GRPO adapter exists (possibly only at a
# checkpoint-N subdir if training crashed before the final save), point at
# the deepest checkpoint we can find.

def find_adapter_dir(base):
    """Return the directory containing adapter_config.json, or None."""
    if not base.exists():
        return None
    if (base / "adapter_config.json").exists():
        return base
    # Look for checkpoint-N subdirs and pick the highest N that has a config.
    cps = []
    for d in base.iterdir():
        if d.is_dir() and d.name.startswith("checkpoint-"):
            tail = d.name.split("-", 1)[1]
            if tail.isdigit() and (d / "adapter_config.json").exists():
                cps.append((int(tail), d))
    if cps:
        return max(cps, key=lambda t: t[0])[1]
    return None

GRPO_ADAPTER_BASE = MODELS_DIR / "grpo_lora"
ADAPTER_DIR = find_adapter_dir(GRPO_ADAPTER_BASE)
HAS_GRPO_ADAPTER = ADAPTER_DIR is not None
print(f"GRPO adapter present: {HAS_GRPO_ADAPTER}  ({ADAPTER_DIR})")

llm_kwargs = dict(
    model=MODEL_ID,
    dtype="bfloat16",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.75,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=128,
    max_num_batched_tokens=16384,
    tensor_parallel_size=TENSOR_PARALLEL,
)
if HAS_GRPO_ADAPTER:
    llm_kwargs.update(enable_lora=True, max_lora_rank=32, max_loras=1)

llm = LLM(**llm_kwargs)

if HAS_GRPO_ADAPTER:
    from vllm.lora.request import LoRARequest
    GRPO_LORA = LoRARequest("grpo", 1, str(ADAPTER_DIR))
    print(f"Reloaded vLLM with adapter from {ADAPTER_DIR}")
else:
    GRPO_LORA = None
    print("No adapter — falling back to base model.")

## 10. Validate the GRPO adapter

Re-run SC on the eval slice with the adapter. Keep it only if no regression.


In [ ]:
if HAS_GRPO_ADAPTER:
    grpo_eval_rows = []
    grpo_eval_path = RESULTS_DIR / "sc_k8_grpo_results.jsonl"
    for idx, item in enumerate(tqdm(eval_items, desc=f"SC k={SC_K} + GRPO")):
        chosen, modal_key, keys = run_sc_one(item, k=SC_K, lora_request=GRPO_LORA)
        grpo_eval_rows.append(dict(
            id=item.get("id"), is_mcq=bool(item.get("options")),
            gold=item["answer"], response=chosen,
            correct=score_one(item, chosen),
            k=SC_K, sample_keys=keys, modal_key=modal_key,
        ))
        if (idx + 1) % 3 == 0 or idx == len(eval_items) - 1:
            write_jsonl(grpo_eval_path, grpo_eval_rows)
    write_jsonl(grpo_eval_path, grpo_eval_rows)
    print_summary(f"SC k={SC_K} + GRPO", grpo_eval_rows)

    base_rows = [json.loads(l) for l in open(RESULTS_DIR / "sc_k8_results.jsonl")]
    base_acc = sum(r["correct"] for r in base_rows) / len(base_rows) * 100
    grpo_acc = sum(r["correct"] for r in grpo_eval_rows) / len(grpo_eval_rows) * 100
    USE_GRPO_FOR_PRIVATE = grpo_acc >= base_acc
    print(f"\nBase model SC k=8 : {base_acc:.2f}%")
    print(f"GRPO  model SC k=8 : {grpo_acc:.2f}%")
    print(f"Decision: {'USE GRPO adapter' if USE_GRPO_FOR_PRIVATE else 'FALL BACK to base'}")
else:
    USE_GRPO_FOR_PRIVATE = False
    print("No adapter — private run will use the base model.")

ACTIVE_LORA = GRPO_LORA if (HAS_GRPO_ADAPTER and USE_GRPO_FOR_PRIVATE) else None

## 11. FINAL - Full `private.jsonl` run

SC + (optional) GRPO adapter. Resumable (skips items already in the output). Aborts cleanly if vLLM dies.


In [ ]:
private_items = [json.loads(l) for l in open(PRIVATE_PATH)]
print(f"Private set: {len(private_items)} items")
print(f"Using LoRA adapter: {ACTIVE_LORA is not None}")

private_path = RESULTS_DIR / "private_final_results.jsonl"

# Resume: skip ids whose existing row has a non-error response.
done_ids = set()
if private_path.exists():
    for l in open(private_path):
        try:
            r = json.loads(l)
            if isinstance(r.get("response"), str) and not r["response"].startswith("[error:"):
                done_ids.add(r["id"])
        except Exception:
            pass
print(f"Already done (valid responses): {len(done_ids)}; "
      f"remaining: {len(private_items) - len(done_ids)}")

# Engine death detection: bail out if too many consecutive items error.
CONSECUTIVE_ERROR_LIMIT = 5
consec_errors = 0
items_to_do = [it for it in private_items if it.get("id") not in done_ids]
n_written = 0

with open(private_path, "a") as f:
    for item in tqdm(items_to_do, desc="Private SC k=8"):
        chosen, _, sample_keys = run_sc_one(item, k=SC_K, lora_request=ACTIVE_LORA)
        # All samples errored ⇒ count toward consecutive errors and skip writing.
        if isinstance(chosen, str) and chosen.startswith("[error:"):
            consec_errors += 1
            if consec_errors >= CONSECUTIVE_ERROR_LIMIT:
                print(f"\nABORTING: {consec_errors} consecutive items errored. "
                      f"vLLM engine likely dead. Re-run this cell to resume.")
                break
            continue
        consec_errors = 0
        f.write(json.dumps(dict(
            id=item.get("id"), is_mcq=bool(item.get("options")),
            response=chosen,
        )) + "\n")
        f.flush()
        n_written += 1

total = sum(1 for _ in open(private_path)) if private_path.exists() else 0
print(f"\nThis run wrote {n_written} new records. Total in file: {total}/{len(private_items)}.")

## 12. Comparison across steps

Quick accuracy comparison.


In [ ]:
def maybe(path):
    p = Path(path)
    if not p.exists(): return None
    return [json.loads(l) for l in open(p)]

print(f"{'Step':<22s} {'MCQ':>8s} {'Free':>8s} {'Overall':>10s}")
print("-" * 52)
for label, path in [
    ("SC k=3",            RESULTS_DIR / "sc_k8_results.jsonl"),
]:
    rs = maybe(path)
    if rs:
        mcq = [r for r in rs if r.get("is_mcq")]
        free = [r for r in rs if not r.get("is_mcq")]
        def acc(rs_): return sum(r["correct"] for r in rs_) / len(rs_) * 100 if rs_ else 0.0
        print(f"{label:<22s} {acc(mcq):>7.2f}% {acc(free):>7.2f}% {acc(rs):>9.2f}%")

sw = maybe(RESULTS_DIR / "param_sweep.jsonl")
if sw:
    sw.sort(key=lambda r: -r["overall_acc"])
    print("\nTop 5 sweep configs:")
    for r in sw[:5]:
        print(f"  {r['config_key']:34s}  overall={r['overall_acc']:5.2f}%")
